# Task 2.2.2: Overview of Popular Object Detection Models
## The basics of popular object detection models, including R-CNN, Fast R-CNN, Faster R-CNN, and YOLO

In [1]:
# Ensuring matplotlib will display correctly:
%matplotlib inline
# Importing required libraries, which include:
# OpenCV library used for the many tasks in computer vision, from loading images, to processing, detecting, shapes, tracking objects etc.:
import cv2
# Numpy is very useful computation library, here we can use it to compute means:
import numpy as np
# We need the following to show images as plots (x and y axes), later we will use it to draw actual plots too:
import matplotlib.pyplot as plt
# We need the following to draw areas of interest on our plots and images:
import matplotlib.patches as patches

In [ ]:
import urllib.request
import tarfile
import os

# Define the URL and local path
model_name = 'faster_rcnn_inception_v2_coco_2018_01_28'
url = f"http://download.tensorflow.org/models/object_detection/{model_name}.tar.gz"
download_path = f"{model_name}.tar.gz"

# Download the model
urllib.request.urlretrieve(url, download_path)

# Extract the tarball
tar = tarfile.open(download_path)
tar.extractall()
tar.close()

# Delete the tarball after extraction
os.remove(download_path)

print("DONE!")

In [ ]:
import pathlib 
import tensorflow as tf
from PIL import Image
from object_detection.utils import label_map_util

# Load the saved model
model_name = 'faster_rcnn_inception_v2_coco_2018_01_28'
model_dir = os.path.join(model_name, "saved_model")
detection_model = tf.saved_model.load(model_dir)

# Load the COCO labels
PATH_TO_LABELS = './mscoco_label_map.pbtxt'
category_index = label_map_util.create_category_index_from_labelmap(PATH_TO_LABELS, use_display_name=True)

# Load an image
def load_image_into_numpy_array(path):
    """Load an image from file into a numpy array."""
    return np.array(Image.open(path))

# Path to the image or video
PATH_TO_TEST_IMAGES_DIR = pathlib.Path('./datasubset')
TEST_IMAGE_PATHS = sorted(list(PATH_TO_TEST_IMAGES_DIR.glob("*.jpg")))

# Detect in image
def run_inference_for_single_image(model, image):
    image = np.asarray(image)
    # The input needs to be a tensor, convert it using `tf.convert_to_tensor`.
    input_tensor = tf.convert_to_tensor(image)
    # The model expects a batch of images, so add an axis with `tf.newaxis`.
    input_tensor = input_tensor[tf.newaxis,...]

    # Run inference
    model_fn = model.signatures['serving_default']
    output_dict = model_fn(input_tensor)

    # All outputs are batches tensors.
    # Convert to numpy arrays, and take index [0] to remove the batch dimension.
    # We're only interested in the first num_detections.
    num_detections = int(output_dict.pop('num_detections'))
    output_dict = {key:value[0, :num_detections].numpy() 
                 for key,value in output_dict.items()}
    output_dict['num_detections'] = num_detections

    # detection_classes should be ints.
    output_dict['detection_classes'] = output_dict['detection_classes'].astype(np.int64)
    return output_dict

print("DONE!")

In [ ]:
# Displaying the images with bounding boxes, labels and confidence scores
def display_image_with_boxes(image_np, detection_boxes, detection_classes, detection_scores, category_index, threshold=0.5):
    """
    Display image with bounding boxes using cv2 and matplotlib.
    
    Args:
        image_np: Numpy array of the image.
        detection_boxes: Detected bounding boxes.
        detection_classes: Class IDs of the detections.
        detection_scores: Scores of the detections.
        category_index: Dictionary mapping class IDs to class names.
        threshold: Score threshold for displaying the bounding box.
    """
    
    # Get image height and width
    im_height, im_width = image_np.shape[:2]

    for i, box in enumerate(detection_boxes):
        if detection_scores[i] > threshold:
            class_name = category_index[detection_classes[i]]['name']
            score = detection_scores[i]

            # Convert box coordinates to pixels
            ymin, xmin, ymax, xmax = box
            (left, right, top, bottom) = (xmin * im_width, xmax * im_width,
                                          ymin * im_height, ymax * im_height)

            # Draw bounding box
            cv2.rectangle(image_np, (int(left), int(top)), (int(right), int(bottom)), (0, 255, 0), 2)

            # Display class label and score
            label = f"{class_name}: {score:.2f}"
            label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            label_rect_top = int(top) - label_size[0][1]
            if label_rect_top < 1:
                label_rect_top = int(top) + label_size[0][1]
            cv2.rectangle(image_np, (int(left), label_rect_top - label_size[0][1]),
                          (int(left) + label_size[0][0], label_rect_top + label_size[1]), (0, 255, 0), -1)
            cv2.putText(image_np, label, (int(left), label_rect_top),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 2)

    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(image_np, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

for image_path in TEST_IMAGE_PATHS:
    image_np = load_image_into_numpy_array(image_path)
    output_dict = run_inference_for_single_image(detection_model, image_np)
    display_image_with_boxes(
      image_np,
      output_dict['detection_boxes'],
      output_dict['detection_classes'],
      output_dict['detection_scores'],
      category_index
    )

Task 1 - Load the pre-trained YOLOv5 model using torch.hub and then detect objects using PyTorch

In [ ]:
import torch

# Load the pre-trained YOLOv5 model using torch.hub

# Set the model to evaluation mode

# Load an image

# Convert the image to RGB format

# Detect objects using PyTorch

height, width, _ = img.shape

# Display the detected objects on the image
for detection in detected_objects:
    x1, y1, x2, y2, confidence, class_id = detection
    if confidence > 0.5:
        # Convert box format from x1,y1,x2,y2 to x,y,w,h
        x = int(x1)
        y = int(y1)
        w = int(x2 - x1)
        h = int(y2 - y1)
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)

cv2.imshow("Image with detected objects", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

plt.figure()
plt.imshow(img)
plt.axis("off")
plt.show()